In [13]:
import pandas as pd 
from IPython.display import display

# 读取数据
movies = pd.read_csv('movies.csv')
display(movies.head(10))

# 创建 genre 矩阵（one-hot 编码）
genre_matrix = movies['genres'].str.get_dummies(sep='|')

# 显示矩阵
display(genre_matrix.head())

# 检查类型
print(f"genre_matrix 类型: {type(genre_matrix)}")
print(f"genre_matrix 形状: {genre_matrix.shape}")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


genre_matrix 类型: <class 'pandas.core.frame.DataFrame'>
genre_matrix 形状: (9125, 20)


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim=cosine_similarity(genre_matrix)
print("相似度矩阵形状:",cosine_sim.shape)

相似度矩阵形状: (9125, 9125)


In [24]:
def get_recommendations(title):
    try:
        idx = movies[movies['title'] == title].index[0]
    except:
        return "电影库没找到这部电影，请检查拼写（包含年份）"
    
    # 获取相似度分数
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # 排除自己，取前5个
    
    # 提取电影索引和分数
    movie_indices = [i[0] for i in sim_scores]
    movie_scores = [i[1] for i in sim_scores]
    
    # 创建结果DataFrame
    recommendations = movies['title'].iloc[movie_indices].reset_index(drop=True)
    scores_df = pd.DataFrame(movie_scores, columns=['cosine_similarity'])
    
    # 合并显示
    result = pd.concat([recommendations, scores_df], axis=1)
    result.columns = ['推荐电影', '相似度分数']
    
    return result

In [25]:
test_movie=movies['title'].iloc[9]
print(f"测试电影:{test_movie}")
recommendations=get_recommendations(test_movie)
print(recommendations)

测试电影:GoldenEye (1995)
                        推荐电影  相似度分数
0        Broken Arrow (1996)    1.0
1         Cliffhanger (1993)    1.0
2  Executive Decision (1996)    1.0
3  Surviving the Game (1994)    1.0
4           Rock, The (1996)    1.0


In [22]:
print(movies[movies['title']==test_movie])

   movieId             title                     genres
9       10  GoldenEye (1995)  Action|Adventure|Thriller


In [26]:
display(movies.head(491))

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
486,542,Son in Law (1993),Comedy|Drama|Romance
487,543,So I Married an Axe Murderer (1993),Comedy|Romance|Thriller
488,544,Striking Distance (1993),Action|Crime
489,546,Super Mario Bros. (1993),Action|Adventure|Children|Comedy|Fantasy|Sci-Fi
